# 1.**🎙️** Dando Voz ao Código com Python

In [ ]:
from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode

RECORD = """
async function record(sec) {
  const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
  const recorder = new MediaRecorder(stream);
  let data = [];
  recorder.ondataavailable = event => data.push(event.data);
  recorder.start();
  await new Promise(resolve => setTimeout(resolve, sec * 1000));
  recorder.stop();
  await new Promise(resolve => recorder.onstop = resolve);
  const blob = new Blob(data);
  const reader = new FileReader();
  reader.readAsDataURL(blob);
  return await new Promise(resolve => reader.onloadend = () => resolve(reader.result));
}
"""

def record(sec=10):
    display(Javascript(RECORD))
    js_result = output.eval_js(f'record({sec})')
    audio = b64decode(js_result.split(',')[1])
    file_name = 'request_audio.wav'
    with open(file_name, 'wb') as f:
        f.write(audio)
    return file_name

print("Gravando...")
record_file = record(10)
display(Audio(record_file))

Gravando...


<IPython.core.display.Javascript object>

# 2. 🧠 Conversão de Voz em Texto com Whisper

In [ ]:
pip install git+https://github.com/openai/whisper.git -q

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
# https://github.com/openai/whisper#available-models-and-languages

import whisper

model = whisper.load_model("small")

# Define o idioma para a transcrição
language = "pt"

# Transcreve o audio gravado anteriormente.
result = model.transcribe(record_file, fp16=False, language=language)
transcription = result["text"]
print(transcription)

 Quais são os riscos das operações bancárias relacionadas à transformação de maturidade, à transferência imperfeita de risco de crédito e à alavancagem?


# **3. 💬 Integração com Hugging Face Transformers (Local)**

In [ ]:
!pip install -q transformers torch

from transformers import pipeline

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gtts 2.5.4 requires click<8.2,>=7.1, but you have click 8.3.1 which is incompatible.


In [ ]:
# Dar voz ao modelo de conversação
chatbot = pipeline("text-generation", model="microsoft/DialoGPT-small")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-small
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
from transformers import pipeline

qa_pipeline = pipeline("question-answering", model="pierreguillou/bert-base-cased-squad-v1.1-portuguese")

context = """
Operações bancárias envolvem riscos relacionados à transformação de maturidade e liquidez,
à transferência imperfeita de risco de crédito e à alavancagem. Esses riscos podem afetar
a estabilidade financeira e a solvência das instituições.
"""

result = qa_pipeline(question=transcription, context=context)
answer = result["answer"]
print("💬 Resposta da IA:", answer)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: pierreguillou/bert-base-cased-squad-v1.1-portuguese
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


💬 Resposta da IA: estabilidade financeira e a solvência das instituições


# 4. 🔊 Gerando Áudio a partir da Resposta (gTTS)

In [ ]:
!pip install gTTS

  Using cached click-8.1.8-py3-none-any.whl.metadata (2.3 kB)
Using cached click-8.1.8-py3-none-any.whl (98 kB)
  Attempting uninstall: click
    Found existing installation: click 8.3.1
    Uninstalling click-8.3.1:
      Successfully uninstalled click-8.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.0 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.


In [ ]:

from gtts import gTTS
from IPython.display import Audio

gtts_object = gTTS(text=answer, lang="pt", slow=False)
response_audio = "/content/response_audio.wav"
gtts_object.save(response_audio)
display(Audio(response_audio, autoplay=True))
